# 3.4. softmax迴歸
:label:`sec_softmax`

在 :numref:`sec_linear_regression`中我們介紹了線性迴歸。
隨後，在 :numref:`sec_linear_scratch`中我們從頭實現線性迴歸。
然後，在 :numref:`sec_linear_concise`中我們使用深度學習框架的高階API簡潔實現線性迴歸。

迴歸可以用於預測*多少*的問題。
比如預測房屋被售出價格，或者棒球隊可能獲得的勝場數，又或者患者住院的天數。

事實上，我們也對*分類*問題感興趣：不是問「多少」，而是問「哪一個」：

* 某個電子郵件是否屬於垃圾郵件資料夾？
* 某個用戶可能*註冊*或*不註冊*訂閱服務？
* 某個圖像描繪的是驢、狗、貓、還是雞？
* 某人接下來最有可能看哪部電影？

通常，機器學習實踐者用*分類*這個詞來描述兩個有微妙差別的問題：
1. 我們只對樣本的「硬性」類別感興趣，即屬於哪個類別；
2. 我們希望得到「軟性」類別，即得到屬於每個類別的機率。
這兩者的界限往往很模糊。其中的一個原因是：即使我們只關心硬類別，我們仍然使用軟類別的模型。

## 3.4.1. 分類問題
:label:`subsec_classification-problem`

我們從一個圖像分類問題開始。
假設每次輸入是一個$2\times2$的灰度圖像。
我們可以用一個標量表示每個像素值，每個圖像對應四個特徵$x_1, x_2, x_3, x_4$。
此外，假設每個圖像屬於類別「貓」「雞」和「狗」中的一個。

接下來，我們要選擇如何表示標籤。
我們有兩個明顯的選擇：最直接的想法是選擇$y \in \{1, 2, 3\}$，
其中整數分別代表$\{\text{狗}, \text{貓}, \text{雞}\}$。
這是在電腦上儲存此類資訊的有效方法。
如果類別間有一些自然順序，
比如說我們試圖預測$\{\text{嬰兒}, \text{兒童}, \text{青少年}, \text{青年人}, \text{中年人}, \text{老年人}\}$，
那麼將這個問題轉變為迴歸問題，並且保留這種格式是有意義的。

但是一般的分類問題並不與類別之間的自然順序有關。
幸運的是，統計學家很早以前就發明了一種表示分類資料的簡單方法：*獨熱編碼*（one-hot encoding）。
獨熱編碼是一個向量，它的分量和類別一樣多。
類別對應的分量設置為1，其他所有分量設置為0。
在我們的例子中，標籤$y$將是一個三維向量，
其中$(1, 0, 0)$對應於「貓」、$(0, 1, 0)$對應於「雞」、$(0, 0, 1)$對應於「狗」：

$$y \in \{(1, 0, 0), (0, 1, 0), (0, 0, 1)\}.$$

## 3.4.2. 網路架構

為了估計所有可能類別的條件概率，我們需要一個有多個輸出的模型，每個類別對應一個輸出。
為了解決線性模型的分類問題，我們需要和輸出一樣多的*仿射函數*（affine function）。
每個輸出對應於它自己的仿射函數。
在我們的例子中，由於我們有4個特徵和3個可能的輸出類別，
我們將需要12個標量來表示權重（帶下標的$w$），
3個標量來表示偏置（帶下標的$b$）。
下面我們為每個輸入計算三個*未規範化的預測*（logit）：$o_1$、$o_2$和$o_3$。

$$
\begin{aligned}
o_1 &= x_1 w_{11} + x_2 w_{12} + x_3 w_{13} + x_4 w_{14} + b_1,\\
o_2 &= x_1 w_{21} + x_2 w_{22} + x_3 w_{23} + x_4 w_{24} + b_2,\\
o_3 &= x_1 w_{31} + x_2 w_{32} + x_3 w_{33} + x_4 w_{34} + b_3.
\end{aligned}
$$

我們可以用神經網路圖 :numref:`fig_softmaxreg`來描述這個計算過程。
與線性迴歸一樣，softmax迴歸也是一個單層神經網路。
由於計算每個輸出$o_1$、$o_2$和$o_3$取決於
所有輸入$x_1$、$x_2$、$x_3$和$x_4$，
所以softmax迴歸的輸出層也是全連接層。

![softmax迴歸是一種單層神經網路](../img/softmaxreg.svg)
:label:`fig_softmaxreg`

為了更簡潔地表達模型，我們仍然使用線性代數符號。
通過向量形式表達為$\mathbf{o} = \mathbf{W} \mathbf{x} + \mathbf{b}$，
這是一種更適合數學和編寫程式碼的形式。
由此，我們已經將所有權重放到一個$3 \times 4$矩陣中。
對於給定數據樣本的特徵$\mathbf{x}$，
我們的輸出是由權重與輸入特徵進行矩陣-向量乘法再加上偏置$\mathbf{b}$得到的。

## 3.4.3. 全連接層的參數開銷
:label:`subsec_parameterization-cost-fc-layers`

正如我們將在後續章節中看到的，在深度學習中，全連接層無處不在。
然而，顧名思義，全連接層是“完全”連接的，可能有很多可學習的參數。
具體來說，對於任何具有$d$個輸入和$q$個輸出的全連接層，
參數開銷為$\mathcal{O}(dq)$，這個數字在實踐中可能高得令人望而卻步。
幸運的是，將$d$個輸入轉換為$q$個輸出的成本可以減少到$\mathcal{O}(\frac{dq}{n})$，
其中超參數$n$可以由我們靈活指定，以在實際應用中平衡參數節約和模型有效性
 :cite:`Zhang.Tay.Zhang.ea.2021`。

## 3.4.4. softmax運算
:label:`subsec_softmax_operation`

現在我們將優化參數以最大化觀測資料的機率。
為了得到預測結果，我們將設置一個閾值，如選擇具有最大機率的標籤。

我們希望模型的輸出$\hat{y}_j$可以視為屬於類$j$的機率，
然後選擇具有最大輸出值的類別$\operatorname*{argmax}_j y_j$作為我們的預測。
例如，如果$\hat{y}_1$、$\hat{y}_2$和$\hat{y}_3$分別為0.1、0.8和0.1，
那麼我們預測的類別是2，在我們的例子中代表“雞”。

然而我們能否將未規範化的預測$o$直接視作我們感興趣的輸出呢？
答案是否定的。
因為將線性層的輸出直接視為機率時存在一些問題：
一方面，我們沒有限制這些輸出數字的總和為1。
另一方面，根據輸入的不同，它們可以為負值。
這些違反了 :numref:`sec_prob`中說的概率基本公理。

要將輸出視為機率，我們必須保證在任何資料上的輸出都是非負的且總和為1。
此外，我們需要一個訓練的目標函數，來激勵模型精確地估計機率。
例如，
在分類器輸出0.5的所有樣本中，我們希望這些樣本是剛好有一半實際上屬於預測的類別。
這個屬性叫做*校準*（calibration）。

社會科學家鄧肯·盧斯於1959年在*選擇模型*（choice model）的理論基礎上
發明的*softmax函數*正是這樣做的：
softmax函數能夠將未規範化的預測變換為非負數並且總和為1，同時讓模型保持
可導的性質。
為了完成這一目標，我們首先對每個未規範化的預測求冪，這樣可以確保輸出非負。
為了確保最終輸出的機率值總和為1，我們再讓每個求冪後的結果除以它們的總和。如下式：

$$\hat{\mathbf{y}} = \mathrm{softmax}(\mathbf{o})\quad \text{其中}\quad \hat{y}_j = \frac{\exp(o_j)}{\sum_k \exp(o_k)}$$
:eqlabel:`eq_softmax_y_and_o`

這裡，對於所有的$j$總有$0 \leq \hat{y}_j \leq 1$。
因此，$\hat{\mathbf{y}}$可以視為一個正確的機率分布。
softmax運算不會改變未規範化的預測$\mathbf{o}$之間的大小次序，只會確定分配給每個類別的機率。
因此，在預測過程中，我們仍然可以用下式來選擇最有可能的類別。

$$
\operatorname*{argmax}_j \hat y_j = \operatorname*{argmax}_j o_j.
$$

儘管softmax是一個非線性函數，但softmax迴歸的輸出仍然由輸入特徵的仿射變換決定。
因此，softmax迴歸是一個*線性模型*（linear model）。

## 3.4.5. 小批量樣本的矢量化
:label:`subsec_softmax_vectorization`

為了提高計算效率並充分利用GPU，我們通常會對小批量樣本資料執行矢量計算。
假設我們讀取了一個批量的樣本$\mathbf{X}$，
其中特徵維度（輸入數量）為$d$，批量大小為$n$。
此外，假設我們在輸出中有$q$個類別。
那麼小批量樣本的特徵為$\mathbf{X} \in \mathbb{R}^{n \times d}$，
權重為$\mathbf{W} \in \mathbb{R}^{d \times q}$，
偏置為$\mathbf{b} \in \mathbb{R}^{1\times q}$。
softmax迴歸的矢量計算表達式為：

$$ \begin{aligned} \mathbf{O} &= \mathbf{X} \mathbf{W} + \mathbf{b}, \\ \hat{\mathbf{Y}} & = \mathrm{softmax}(\mathbf{O}). \end{aligned} $$
:eqlabel:`eq_minibatch_softmax_reg`

相對於一次處理一個樣本，
小批量樣本的矢量化加快了$\mathbf{X}和\mathbf{W}$的矩陣-向量乘法。
由於$\mathbf{X}$中的每一行代表一個資料樣本，
那麼softmax運算可以*按行*（rowwise）執行：
對於$\mathbf{O}$的每一行，我們先對所有項進行冪運算，然後通過求和對它們進行標準化。
在 :eqref:`eq_minibatch_softmax_reg`中，
$\mathbf{X} \mathbf{W} + \mathbf{b}$的求和會使用廣播機制，
小批量的未規範化預測$\mathbf{O}$和輸出機率$\hat{\mathbf{Y}}$
都是形狀為$n \times q$的矩陣。

## 3.4.6. 損失函數

接下來，我們需要一個損失函數來度量預測的效果。
我們將使用最大似然估計，這與線性迴歸
（ :numref:`subsec_normal_distribution_and_squared_loss`）
中的方法相同。

### 3.4.6.1. 對數似然

softmax函數給出了一個向量$\hat{\mathbf{y}}$，
我們可以將其視為“給定任意輸入$\mathbf{x}$的每個類的條件機率”。
例如，$\hat{y}_1$=$P(y=\text{貓} \mid \mathbf{x})$。
假設整個資料集$\{\mathbf{X}, \mathbf{Y}\}$具有$n$個樣本，
其中索引$i$的樣本由特徵向量$\mathbf{x}^{(i)}$和獨熱標籤向量$\mathbf{y}^{(i)}$組成。
我們可以將估計值與實際值進行比較：

$$
P(\mathbf{Y} \mid \mathbf{X}) = \prod_{i=1}^n P(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)}).
$$

根據最大似然估計，我們最大化$P(\mathbf{Y} \mid \mathbf{X})$，相當於最小化負對數似然：

$$
-\log P(\mathbf{Y} \mid \mathbf{X}) = \sum_{i=1}^n -\log P(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)})
= \sum_{i=1}^n l(\mathbf{y}^{(i)}, \hat{\mathbf{y}}^{(i)}),
$$

其中，對於任何標籤$\mathbf{y}$和模型預測$\hat{\mathbf{y}}$，損失函數為：

$$ l(\mathbf{y}, \hat{\mathbf{y}}) = - \sum_{j=1}^q y_j \log \hat{y}_j. $$
:eqlabel:`eq_l_cross_entropy`

在本節稍後的內容會講到， :eqref:`eq_l_cross_entropy`中的損失函數
通常被稱為*交叉熵損失*（cross-entropy loss）。
由於$\mathbf{y}$是一個長度為$q$的獨熱編碼向量，
所以除了一個項以外的所有項$j$都消失了。
由於所有$\hat{y}_j$都是預測的概率，所以它們的對數永遠不會大於$0$。
因此，如果正確地預測實際標籤，即如果實際標籤$P(\mathbf{y} \mid \mathbf{x})=1$，
則損失函數不能進一步最小化。
注意，這往往是不可能的。
例如，資料集中可能存在標籤噪聲（比如某些樣本可能被誤標），
或輸入特徵沒有足夠的資訊來完美地對每一個樣本分類。

### 3.4.6.2. softmax及其導數
:label:`subsec_softmax_and_derivatives`

由於softmax和相關的損失函數很常見，
因此我們需要更好地理解它的計算方式。
將 :eqref:`eq_softmax_y_and_o`代入損失 :eqref:`eq_l_cross_entropy`中。
利用softmax的定義，我們得到：

$$
\begin{aligned}
l(\mathbf{y}, \hat{\mathbf{y}}) &=  - \sum_{j=1}^q y_j \log \frac{\exp(o_j)}{\sum_{k=1}^q \exp(o_k)} \\
&= \sum_{j=1}^q y_j \log \sum_{k=1}^q \exp(o_k) - \sum_{j=1}^q y_j o_j\\
&= \log \sum_{k=1}^q \exp(o_k) - \sum_{j=1}^q y_j o_j.
\end{aligned}
$$

考慮相對於任何未規範化的預測$o_j$的導數，我們得到：

$$
\partial_{o_j} l(\mathbf{y}, \hat{\mathbf{y}}) = \frac{\exp(o_j)}{\sum_{k=1}^q \exp(o_k)} - y_j = \mathrm{softmax}(\mathbf{o})_j - y_j.
$$

换句话说，導數是我們softmax模型分配的概率與實際發生情況（由獨熱標籤向量表示）之間的差異。
從這個意義上講，這與我們在迴歸中看到的非常相似，
其中梯度是觀測值$y$和估計值$\hat{y}$之間的差異。
這不是巧合，在任何指數族分布模型中
（參見[書附錄中關於數學分布的一節](https://d2l.ai/chapter_appendix-mathematics-for-deep-learning/distributions.html)），
對數似然的梯度正是由此得出的。
這使梯度計算在實踐中變得容易很多。

### 3.4.6.3. 交叉熵損失

現在讓我們考慮整個結果分佈的情況，即觀測到的不僅僅是一個結果。
對於標籤$\mathbf{y}$，我們可以使用與以前相同的表示形式。
唯一的區別是，現在我們用一個概率向量表示，如$(0.1, 0.2, 0.7)$，
而不是僅包含二元項的向量$(0, 0, 1)$。
我們使用 :eqref:`eq_l_cross_entropy`來定義損失$l$，
它是所有標籤分佈的預期損失值。
此損失稱為*交叉熵損失*（cross-entropy loss），它是分類問題最常用的損失之一。
本節我們將通過介紹信息論基礎來理解交叉熵損失。
如果想了解更多信息論的細節，請進一步參考
[書附錄中關於信息論的一節](https://d2l.ai/chapter_appendix-mathematics-for-deep-learning/information-theory.html)。

## 3.4.7. 信息論基礎
:label:`subsec_info_theory_basics`

*信息論*（information theory）涉及編碼、解碼、發送以及盡可能簡潔地處理信息或資料。

### 3.4.7.1. 熵

信息論的核心思想是量化資料中的信息內容。
在信息論中，該數值被稱為分佈$P$的*熵*（entropy）。可以通過以下方程得到：

$$H[P] = \sum_j - P(j) \log P(j).$$
:eqlabel:`eq_softmax_reg_entropy`

信息論的基本定理之一指出，為了對從分佈$p$中隨機抽取的資料進行編碼，
我們至少需要$H[P]$“納特（nat）”對其進行編碼。
“納特”相當於*比特*（bit），但是對數底為$e$而不是2。因此，一個納特是$\frac{1}{\log(2)} \approx 1.44$比特。

### 3.4.7.2. 信息量

壓縮與預測有什麼關係呢？
想象一下，我們有一個要壓縮的資料流。
如果我們很容易預測下一個資料，那麼這個資料就很容易壓縮。
為什麼呢？
舉一個極端的例子，假如資料流中的每個資料完全相同，這會是一個非常無聊的資料流。
由於它們總是相同的，我們總是知道下一個資料是什麼。
所以，為了傳遞資料流的內容，我們不必傳遞任何資訊。
也就是說，“下一個資料是xx”這個事件毫無信息量。

但是，如果我們不能完全預測每一個事件，那麼我們有時可能會感到"驚異"。
克勞德·香農決定用信息量$\log \frac{1}{P(j)} = -\log P(j)$來量化這種驚異程度。
在觀察一個事件$j$時，並賦予它（主觀）概率$P(j)$。
當我們賦予一個事件較低的概率時，我們的驚異會更大，該事件的信息量也就更大。
在 :eqref:`eq_softmax_reg_entropy`中定義的熵，
是當分配的概率真正匹配資料生成過程時的*信息量的期望*。

### 3.4.7.3. 重新審視交叉熵

如果把熵$H(P)$想象為“知道真實概率的人所經歷的驚異程度”，那麼什麼是交叉熵？
交叉熵*從*$P$*到*$Q$，記為$H(P, Q)$。
我們可以將交叉熵想象為“主觀概率為$Q$的觀察者在看到根據概率$P$生成的資料時的預期驚異”。
當$P=Q$時，交叉熵達到最低。
在這種情況下，從$P$到$Q$的交叉熵是$H(P, P)= H(P)$。

簡而言之，我們可以從兩方面來考慮交叉熵分類目標：
（i）最大化觀測資料的似然；（ii）最小化傳達標籤所需的驚異。

## 3.4.8. 模型預測和評估

在訓練softmax迴歸模型後，給出任何樣本特徵，我們可以預測每個輸出類別的概率。
通常我們使用預測概率最高的類別作為輸出類別。
如果預測與實際類別（標籤）一致，則預測是正確的。
在接下來的實驗中，我們將使用*精度*（accuracy）來評估模型的性能。
精度等於正確預測數與預測總數之間的比率。

## 3.4.9. 小結

* softmax運算獲取一個向量並將其映射為概率。
* softmax迴歸適用於分類問題，它使用了softmax運算中輸出類別的概率分佈。
* 交叉熵是一個衡量兩個概率分佈之間差異的很好的度量，它測量給定模型編碼資料所需的比特數。

## 3.4.10. 練習

1. 我们可以更深入地探討指數族與softmax之間的聯繫。
    1. 計算softmax交叉熵損失$l(\mathbf{y},\hat{\mathbf{y}})$的二階導數。
    1. 計算$\mathrm{softmax}(\mathbf{o})$給出的分佈方差，並與上面計算的二階導數匹配。
1. 假設我們有三個類發生的概率相等，即概率向量是$(\frac{1}{3}, \frac{1}{3}, \frac{1}{3})$。
    1. 如果我們嘗試為它設計二進制代碼，會有什麼問題？
    1. 請設計一個更好的代碼。提示：如果我們嘗試編碼兩個獨立的觀察結果會發生什麼？如果我們聯合編碼$n$個觀察值怎麼辦？
1. softmax是上面介紹的映射的誤稱（雖然深度學習領域中很多人都使用這個名字）。真正的softmax被定義為$\mathrm{RealSoftMax}(a, b) = \log (\exp(a) + \exp(b))$。
    1. 證明$\mathrm{RealSoftMax}(a, b) > \mathrm{max}(a, b)$。
    1. 證明$\lambda^{-1} \mathrm{RealSoftMax}(\lambda a, \lambda b) > \mathrm{max}(a, b)$成立，前提是$\lambda > 0$。
    1. 證明對於$\lambda \to \infty$，有$\lambda^{-1} \mathrm{RealSoftMax}(\lambda a, \lambda b) \to \mathrm{max}(a, b)$。
    1. soft-min會是什麼樣子？
    1. 將其擴展到兩個以上的數字。

[Discussions](https://discuss.d2l.ai/t/1785)


## 3.4.10. 回答

1. 我们可以更深入地探討指數族與softmax之間的聯繫。
    1. 計算softmax交叉熵損失$l(\mathbf{y},\hat{\mathbf{y}})$的二階導數。
    1. 計算$\mathrm{softmax}(\mathbf{o})$給出的分佈方差，並與上面計算的二階導數匹配。

    Ans:
以下的內容將分成三個部分：

1. **複習：softmax及其交叉熵損失函式**  
2. **計算：softmax交叉熵損失 $l(\mathbf{y}, \hat{\mathbf{y}})$ 的二階導數**  
3. **匹配：softmax給出的機率分佈的方差（或協方差矩陣）如何對應到上面計算的二階導數**  



## 1. 複習：softmax及其交叉熵損失

給定一組 logits（也稱為未經歸一化的分數）$\mathbf{o} = (o_1, o_2, \dots, o_C)$，softmax函式定義為

$$
\hat{y}_i 
= \mathrm{softmax}(\mathbf{o})_i
= \frac{e^{o_i}}{\sum_{k=1}^{C} e^{o_k}}.
$$

若真實標籤（one-hot）向量為 $\mathbf{y} = (y_1, y_2, \dots, y_C)$，其只有在正確類別 $i^*$ 上為1，其他維度為0。則常見的**交叉熵損失**可寫成

$$
l(\mathbf{y}, \hat{\mathbf{y}})
= - \sum_{i=1}^{C} y_i \log \hat{y}_i 
= - \sum_{i=1}^{C} y_i \log \left(\frac{e^{o_i}}{\sum_{k} e^{o_k}}\right).
$$

由於 $\mathbf{y}$ 為one-hot，這個式子也經常寫成

$$
l(\mathbf{y}, \hat{\mathbf{y}})
= - \log \hat{y}_{i^*}
= - \bigl(o_{i^*} - \log \sum_{k} e^{o_k}\bigr).
$$

以下推導中，我們會將偏微分都視為相對於$\mathbf{o} = (o_1, \dots, o_C)$做運算。

---

## 2. 計算：softmax交叉熵損失的二階導數

### 2.1 一階導數

先從一階導數開始。令

$$
l(\mathbf{y}, \hat{\mathbf{y}})
= - \sum_{i=1}^{C} y_i \log \hat{y}_i.
$$

對於每個logit $o_j$，我們要計算

$$
\frac{\partial l}{\partial o_j}.
$$

由於 $\hat{y}_i = \frac{e^{o_i}}{\sum_{k} e^{o_k}}$，可以先寫出

$$
\frac{\partial \hat{y}_i}{\partial o_j}
= \hat{y}_i 
\Bigl(\delta_{ij} - \hat{y}_j \Bigr),
$$
其中 $\delta_{ij}$ 為Kronecker delta（當 $i=j$ 時為1，否則為0）。

而

$$
\frac{\partial l}{\partial o_j}
= - \sum_{i=1}^{C} y_i \frac{1}{\hat{y}_i} \frac{\partial \hat{y}_i}{\partial o_j}.
$$

將上式合併可以得到

$$
\frac{\partial l}{\partial o_j}
= - \sum_{i=1}^{C} y_i \frac{1}{\hat{y}_i} 
\, \hat{y}_i \bigl(\delta_{ij} - \hat{y}_j\bigr)
= - \sum_{i=1}^{C} y_i \bigl(\delta_{ij} - \hat{y}_j\bigr).
$$

由於 $\sum_{i=1}^{C} y_i = 1$ （one-hot），以及 $\sum_{i=1}^{C} y_i \delta_{ij} = y_j$，所以可以進一步化簡為

$$
\frac{\partial l}{\partial o_j}
= - \Bigl(y_j - y_j \hat{y}_j - \sum_{i=1}^{C} y_i(-\hat{y}_j)\Bigr).
$$

不過更常見更簡潔的形式，若 $\mathbf{y}$ 為one-hot、真實類別為 $i^*$，則

$$
\frac{\partial l}{\partial o_j}
= \hat{y}_j - y_j.
$$

（因為只有在 $j = i^*$ 時 $y_j=1$，否則0。）

> 在多類(one-hot)分類問題中，最常見的結果是：  
> $\nabla_{\mathbf{o}}\,l = (\hat{\mathbf{y}} - \mathbf{y}).$

---

### 2.2 二階導數

接著，我們就能進一步算**二階導數**（即Hessian矩陣）。對於每一對 \((j, k)\)，定義

$$
\frac{\partial^2 l}{\partial o_k \partial o_j}
= \frac{\partial}{\partial o_k} 
\left(\frac{\partial l}{\partial o_j}\right)
= \frac{\partial}{\partial o_k} (\hat{y}_j - y_j).
$$

由於 $y_j$ 是常數（one-hot標籤），故只有 $\hat{y}_j$ 需要被微分。先回憶

$$
\frac{\partial \hat{y}_j}{\partial o_k}
= \hat{y}_j (\delta_{jk} - \hat{y}_k).
$$

因此

$$
\frac{\partial^2 l}{\partial o_k \partial o_j}
= \frac{\partial}{\partial o_k} \hat{y}_j
= \hat{y}_j (\delta_{jk} - \hat{y}_k).
$$

也就是

$$
\frac{\partial^2 l}{\partial o_k \partial o_j}
= 
\begin{cases}
\hat{y}_j (1 - \hat{y}_j), & \text{若 } j = k,\\
-\,\hat{y}_j \,\hat{y}_k, & \text{若 } j \neq k.
\end{cases}
$$

若把這整個二階導數（對所有 $j,k$）整理成一個矩陣，就能得到一個 $C \times C$ 的Hessian：

$$
\nabla^2_{\mathbf{o}}\, l
= 
\begin{bmatrix}
\hat{y}_1(1 - \hat{y}_1) & -\hat{y}_1 \hat{y}_2 & \dots & -\hat{y}_1 \hat{y}_C \\
-\hat{y}_2 \hat{y}_1 & \hat{y}_2(1 - \hat{y}_2) & \dots & -\hat{y}_2 \hat{y}_C \\
\vdots & \vdots & \ddots & \vdots \\
-\hat{y}_C \hat{y}_1 & -\hat{y}_C \hat{y}_2 & \dots & \hat{y}_C(1 - \hat{y}_C)
\end{bmatrix}.
$$

此矩陣可視為

$$
\mathrm{diag}(\hat{\mathbf{y}}) 
- \hat{\mathbf{y}} \hat{\mathbf{y}}^{\mathsf{T}},
$$

其中 $\mathrm{diag}(\hat{\mathbf{y}})$ 表示將 $\hat{\mathbf{y}}$ 放在對角線上形成的對角矩陣。

---

## 3. 匹配：softmax給出的分佈「方差/協方差」如何對應Hessian

### 3.1 softmax分佈的協方差矩陣

若我們把 $\hat{\mathbf{y}}$ 視為一個機率分佈 $(\hat{y}_1, \hat{y}_2, \dots, \hat{y}_C)$，它的協方差矩陣 $\boldsymbol{\Sigma}$ 一般定義為

$$
\boldsymbol{\Sigma}
= \mathbb{E}\bigl[(\mathbf{X} - \mathbb{E}[\mathbf{X}])(\mathbf{X} - \mathbb{E}[\mathbf{X}])^{\mathsf{T}}\bigr],
$$
其中 $\mathbf{X}$ 服從該機率分佈（即 $\Pr(X = i) = \hat{y}_i$）。對於多類別離散分佈（one-hot狀態），有名的結果是

$$
\boldsymbol{\Sigma}
= \mathrm{diag}(\hat{\mathbf{y}}) 
- \hat{\mathbf{y}} \hat{\mathbf{y}}^{\mathsf{T}}.
$$

- 對角線元素 $\Sigma_{ii} = \hat{y}_i(1 - \hat{y}_i)$ 反映該類別的「方差」。
- 非對角元素 $\Sigma_{ij} = -\,\hat{y}_i \hat{y}_j$，反映該類別間的「協方差」。

### 3.2 與Hessian的對應

由上面(2.2)中的二階導數可知：

$$
\nabla^2_{\mathbf{o}}\, l
= 
\mathrm{diag}(\hat{\mathbf{y}}) 
- \hat{\mathbf{y}} \hat{\mathbf{y}}^{\mathsf{T}}
= \boldsymbol{\Sigma},
$$

這個$\boldsymbol{\Sigma}$就是softmax分佈的協方差矩陣。或更嚴謹地說，在「最大似然估計」/「負對數似然」的觀點下，一個指數族分佈的負對數似然函式的Hessian（相對於其自然參數參數化）就是該分佈的協方差矩陣，也就是常說的Fisher information與分佈方差之間的關係。

換句話說，**softmax+交叉熵損失函式的Hessian** 恰好 **= 由softmax給出的機率分佈的協方差矩陣**。

---

## 總結

1. **softmax交叉熵損失**  
  $$
   l(\mathbf{y}, \hat{\mathbf{y}})
   = - \sum_{i} y_i \log \hat{y}_i
   $$
   若$\mathbf{y}$為one-hot，則其對logit $\mathbf{o}$ 的**一階導數**為 $\hat{\mathbf{y}} - \mathbf{y}$。

2. **二階導數（Hessian）**  
   $$
   \nabla^2_{\mathbf{o}}\, l
   = \mathrm{diag}(\hat{\mathbf{y}}) 
   - \hat{\mathbf{y}} \,\hat{\mathbf{y}}^{\mathsf{T}}.
   $$

3. **softmax分佈的協方差**  
   $$
   \boldsymbol{\Sigma}
   = \mathrm{diag}(\hat{\mathbf{y}}) 
   - \hat{\mathbf{y}} \,\hat{\mathbf{y}}^{\mathsf{T}},
   $$
   正好與上式相符。

從更高層次的觀點來看，softmax分佈（多項式分佈的一種）是指數族的一個例子，而對數似然函式的Hessian和分佈的協方差矩陣之間有對應關係，正好解釋了上述推導中「二階導數 = 機率分佈的變異結構」這項重要性質。


2. 假設我們有三個類發生的概率相等，即概率向量是$(\frac{1}{3}, \frac{1}{3}, \frac{1}{3})$。
    1. 如果我們嘗試為它設計二進制代碼，會有什麼問題？
    1. 請設計一個更好的代碼。提示：如果我們嘗試編碼兩個獨立的觀察結果會發生什麼？如果我們聯合編碼$n$個觀察值怎麼辦？

    Ans:
    1. 三個類（機率各$1/3$）若直接用二進制編碼，每個符號都想用固定長度編碼時，無法達到理論最佳的 $\log_2(3)\approx1.585$ bits；要嘛需要 2 bits 才能表示三種情況，導致不夠「壓縮」。

    2.  
    - **聯合編碼**（如把兩個或多個觀察結果一起編碼）能產生 $3^n$ 種可能，再用二進制對這些可能做編碼，平均起來就能更接近理想的 $\log_2(3)$ bits/符號。  
    - 例如對 2 個觀察結果聯合編碼，有 9 種（$3^2=9$）情況，編碼長度約 $\log_2(9)=3.17$ bits，平均每個符號就能壓到約 1.585 bits，比直接單獨編碼更有效率。

3. softmax是上面介紹的映射的誤稱（雖然深度學習領域中很多人都使用這個名字）。真正的softmax被定義為$\mathrm{RealSoftMax}(a, b) = \log (\exp(a) + \exp(b))$。
    1. 證明$\mathrm{RealSoftMax}(a, b) > \mathrm{max}(a, b)$。
    1. 證明$\lambda^{-1} \mathrm{RealSoftMax}(\lambda a, \lambda b) > \mathrm{max}(a, b)$成立，前提是$\lambda > 0$。
    1. 證明對於$\lambda \to \infty$，有$\lambda^{-1} \mathrm{RealSoftMax}(\lambda a, \lambda b) \to \mathrm{max}(a, b)$。
    1. soft-min會是什麼樣子？
    1. 將其擴展到兩個以上的數字。

    Ans:

    1. **RealSoftMax(a,b) = log(e^a + e^b)；證明 RealSoftMax(a,b) > max(a,b)**  
    - 令 m = max(a,b)。則  
        RealSoftMax(a,b) 
        = log(e^m (1 + e^(min(a,b)-m)))
        = m + log(1 + e^(min(a,b)-m))。
        由於 min(a,b)-m ≤ 0 且「log(1 + ·)」大於0（除非a=b），故  
        RealSoftMax(a,b) > m = max(a,b)。  
    
    2. **λ^(-1)RealSoftMax(λa,λb) > max(a,b)，λ>0**  
    - 同理，令 m=max(a,b)：  
        λ^(-1) log(e^(λa) + e^(λb)) 
        = λ^(-1)(λm + log(1 + e^(λ(min(a,b)-m))))
        = m + (1/λ)log(1 + e^(λ(min(a,b)-m)))。
        後面那項仍大於0，故結果大於max(a,b)。  
    
    3. **當 λ → ∞ 時，λ^(-1)RealSoftMax(λa,λb)→ max(a,b)**  
    - 若 a > b，則 min(a,b)-m = b-a < 0，e^(λ(b-a))→ 0，所以  
        lim(λ → ∞)(m + (1/λ)log(1 + e^(λ(b-a)))) 
        = m = a。
        反之亦然，因此極限即max(a,b)。  
    
    4. **soft-min 的定義**  
    - 可類比為 RealSoftMin(a,b) = -RealSoftMax(-a, -b)。  
    
    5. **擴展到多個數字**  
    - RealSoftMax(x₁,x₂,...,xₙ) = log(e^(x₁) + e^(x₂) + ... + e^(xₙ))。  
    - 性質皆類似，可用同樣手法證明對應的不等式與極限。  

